In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/02_dim_procedure
# Purpose  : SCD Type 1 for medical procedure codes (SNOMED-CT, 170 distinct codes).
#            apply_changes handles deduplication internally using
#            keys + sequence_by, eliminating the need for an explicit dropDuplicates.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_procedure_changes")
def v_procedure_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.procedures")
    return df.select(
        F.xxhash64("procedure_code").alias("procedure_key"),
        "procedure_code",
        "procedure_description",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_procedure")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_procedure",
    source="v_procedure_changes",
    keys=["procedure_code"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)